## Keras ke saath Neural Networks ka sabse simple introduction

> Yeh notebook [AI for Beginners Curricula](http://github.com/microsoft/ai-for-beginners) ka hissa hai. Poora learning material dekhne ke liye repository par jaayein.

### Neural Frameworks

Neural network ko train karne ke liye bahut saare frameworks available hain. Lekin agar aap jaldi shuru karna chahte hain aur yeh jaanne mein zyada gehraai mein nahi jaana chahte ki cheezein andar se kaise kaam karti hain - toh aapko [Keras](https://keras.io/) use karne par vichaar karna chahiye. Yeh chhota sa tutorial aapko shuruaat karne mein help karega, aur agar aap gehraai mein samajhna chahte hain ki cheezein kaise kaam karti hain - toh [Introduction to Tensorflow and Keras](IntroKerasTF.ipynb) notebook dekhiye.

### Setup taiyaar karna

Keras Tensorflow 2.x framework ka hissa hai. Chalo pehle check kar lete hain ki hamare paas Tensorflow ka version 2.x.x install hai:
```
pip install tensorflow
```
ya
```
conda install tensorflow
```

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt
print(f'Tensorflow version = {tf.__version__}')
print(f'Keras version = {keras.__version__}')

## Basic Concepts: Tensor

**Tensor** ek multi-dimensional array hota hai. Alag-alag tarah ke data ko represent karne ke liye tensor use karna bahut convenient hai:
* 400x400 - black and white image
* 400x400x3 - color image
* 16x400x400x3 - 16 color images ka minibatch
* 25x400x400x3 - ek second ka video, 25 frames per second ke saath
* 8x25x400x400x3 - 8 one-second videos ka minibatch

Tensors input/output data ko represent karne ka ek convenient tarika dete hain, aur saath hi neural network ke andar ke weights ko bhi.

## Sample Problem

Chalo ek binary classification problem lete hain. Aisi problem ka ek accha example hoga tumor ko uske size aur age ke basis par malignant (ghaatak) ya benign (saumya) mein classify karna. Shuru karte hain kuch sample data generate karke:

In [ ]:
np.random.seed(0) # pick the seed for reproducibility - change it to explore the effects of random variations

n = 100
X, Y = make_classification(n_samples = n, n_features=2,
                           n_redundant=0, n_informative=2, flip_y=0.05,class_sep=1.5)
X = X.astype(np.float32)
Y = Y.astype(np.int32)

split = [ 70*n//100 ]
train_x, test_x = np.split(X, split)
train_labels, test_labels = np.split(Y, split)

In [ ]:
def plot_dataset(features, labels, W=None, b=None):
    # prepare the plot
    fig, ax = plt.subplots(1, 1)
    ax.set_xlabel('$x_i[0]$ -- (feature 1)')
    ax.set_ylabel('$x_i[1]$ -- (feature 2)')
    colors = ['r' if l else 'b' for l in labels]
    ax.scatter(features[:, 0], features[:, 1], marker='o', c=colors, s=100, alpha = 0.5)
    if W is not None:
        min_x = min(features[:,0])
        max_x = max(features[:,1])
        min_y = min(features[:,1])*(1-.1)
        max_y = max(features[:,1])*(1+.1)
        cx = np.array([min_x,max_x],dtype=np.float32)
        cy = (0.5-W[0]*cx-b)/W[1]
        ax.plot(cx,cy,'g')
        ax.set_ylim(min_y,max_y)
    fig.show()

In [ ]:
plot_dataset(train_x, train_labels)

## Data ko Normalize karna

Training shuru karne se pehle, input features ko standard range [0,1] (ya [-1,1]) mein laana ek common practice hai. Iske exact reasons par hum course mein aage baat karenge, lekin short mein reason yeh hai. Hum chahte hain ki hamare network se guzarne waali values na bahut badi hon, na bahut chhoti, aur aam taur par hum maante hain ki saari values 0 ke aas-paas ek chhoti range mein rahein. Isliye hum weights ko chhote random numbers se initialize karte hain aur signals ko usi range mein banaye rakhte hain.

Data normalize karte waqt, hume minimum value ghatani hoti hai aur range se divide karna hota hai. Hum minimum value aur range ko training data se calculate karte hain, aur phir test/validation datasets ko training set se mili usi min/range values se normalize karte hain. Aisa isliye kyunki real life mein hume sirf training set ka hi pata hoga, aur network se prediction ke liye aane waali saari nayi values ka pata nahi hoga. Kabhi-kabhi, nayi value [0,1] range ke bahar ho sakti hai, lekin yeh zyada important nahi hai.

In [ ]:
train_x_norm = (train_x-np.min(train_x,axis=0)) / (np.max(train_x,axis=0)-np.min(train_x,axis=0))
test_x_norm = (test_x-np.min(train_x,axis=0)) / (np.max(train_x,axis=0)-np.min(train_x,axis=0))

## Single-layer Network (Perceptron) ko Train karna

Kai cases mein, neural network layers ki ek series hota hai. Ise Keras mein `Sequential` model use karke is tarah define kiya ja sakta hai:

In [ ]:
model = keras.models.Sequential()
model.add(keras.Input(shape=(2,)))
model.add(keras.layers.Dense(1))
model.add(keras.layers.Activation(keras.activations.sigmoid))

model.summary()

Yahan, hum pehle model banate hain, aur phir usmein layers add karte hain:
* Pehla `Input` layer (jo technically layer nahi hai) network ke input size ki specification batata hai.
* `Dense` layer asli perceptron hai jismein trainable weights hote hain.
* Aakhir mein, ek layer hoti hai jismein *sigmoid* `Activation` function hota hai, jo network ke result ko 0-1 range mein laane ke liye use hota hai (taaki ise probability banaya ja sake).

Input size aur activation function ko seedhe `Dense` layer mein bhi specify kiya ja sakta hai, jisse code chhota aur simple ho jaata hai:

In [ ]:
model = keras.models.Sequential()
model.add(keras.layers.Dense(1,input_shape=(2,),activation='sigmoid'))
model.summary()

Model ko train karne se pehle, hume ise **compile** karna hoga, yaani yeh specify karna hoga:
* **Loss function**, jo define karta hai ki loss kaise calculate hoga. Kyunki hamare paas two-class classification problem hai, hum *binary cross-entropy loss* use karenge.
* Use karne ke liye **optimizer**. Sabse simple option hoga *stochastic gradient descent* ke liye `sgd` use karna, ya aap `adam` jaise zyada advanced optimizer bhi use kar sakte hain.
* **Metrics**, jinhe hum apni training ki success measure karne ke liye use karna chahte hain. Kyunki yeh classification task hai, ek accha metric `Accuracy` (ya short mein `acc`) hoga.

Hum loss, metrics aur optimizer ko ya toh strings ke roop mein specify kar sakte hain, ya Keras framework ke kuch objects dekar. Hamare example mein, hume apne model ki learning speed ko fine-tune karne ke liye `learning_rate` parameter specify karna hoga, isliye hum Keras SGD optimizer ka poora naam dete hain.

In [ ]:
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.2),loss='binary_crossentropy',metrics=['acc'])

Model compile karne ke baad, hum actual training `fit` method ko call karke kar sakte hain. Sabse important parameters yeh hain:
* `x` aur `y` respectively training data, yaani features aur labels specify karte hain
* Agar hum chahte hain ki har epoch par validation ho, toh hum `validation_data` parameter specify kar sakte hain, jo features aur labels ka ek tuple hoga
* `epochs` epochs ki number specify karta hai
* Agar hum chahte hain ki training minibatches mein ho, toh hum `batch_size` parameter specify kar sakte hain. Aap data ko manually pehle se batch karke `x`/`y`/`validation_data` mein pass bhi kar sakte hain, us case mein aapko `batch_size` ki zaroorat nahi padegi

In [ ]:
model.fit(x=train_x_norm,y=train_labels,validation_data=(test_x_norm,test_labels),epochs=10,batch_size=1)

Aap alag-alag training parameters ke saath experiment kar sakte hain aur dekh sakte hain ki woh training ko kaise affect karte hain:
* Agar `batch_size` bahut bada set kiya gaya hai (ya bilkul specify hi nahi kiya gaya), toh training kam stable ho sakti hai, kyunki low-dimensional data ke saath chhote batch sizes har specific case ke liye gradient ki direction zyada accurately dete hain.
* Bahut zyada `learning_rate` se overfitting ho sakti hai, ya results kam stable ho sakte hain, jabki bahut kam `learning_rate` ka matlab hai ki results paane ke liye zyada epochs lagenge.

> Dhyan rakhiye ki aap network ko aur train karne ke liye `fit` function ko lagaataar kai baar call kar sakte hain. Agar aap training ko shuru se start karna chahte hain - toh aapko model definition waali cell ko dobara run karna hoga.

Yeh confirm karne ke liye ki hamari training successful rahi, chalo us line ko plot karte hain jo do classes ko alag karti hai. Separation line ki equation $W\times x + b = 0.5$ se define hoti hai.

In [ ]:
plot_dataset(train_x,train_labels,model.layers[0].weights[0],model.layers[0].weights[1])

## Training Graph ko Plot karna

`fit` function ek `history` object return karta hai, jise har epoch par loss aur metrics dekhne ke liye use kiya ja sakta hai. Neeche diye gaye example mein, hum chhote learning rate ke saath training dobara shuru karenge aur dekhenge ki loss aur accuracy kaise behave karte hain.

> **Dhyan dein** ki hum `Sequential` model define karne ke liye thoda alag syntax use kar rahe hain. Layers ko ek-ek karke `add` karne ke bajaye, hum model banate waqt hi layers ki list specify kar sakte hain - yeh thoda chhota syntax hai, aur ho sakta hai aapko ise use karna pasand aaye.

In [ ]:
model = keras.models.Sequential([
    keras.layers.Dense(1,input_shape=(2,),activation='sigmoid')])
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.05),loss='binary_crossentropy',metrics=['acc'])
hist = model.fit(x=train_x_norm,y=train_labels,validation_data=(test_x_norm,test_labels),epochs=10,batch_size=1)

In [ ]:
plt.plot(hist.history['acc'])
plt.plot(hist.history['val_acc'])

## Multi-class Classification

Agar aapko multi-class classification ki problem solve karni hai, toh aapke network mein ek se zyada outputs honge - jo classes ki sankhya $C$ ke barabar honge. Har output ek di gayi class ki probability dikhayega.

> Dhyan rakhiye ki aap do outputs waale network se isi tarike se binary classification bhi kar sakte hain. Yahi hum ab demonstrate karenge.

Jab hum expect karte hain ki network probabilities ka ek set $p_1,\dots, p_C$ output karega, toh hume ensure karna hoga ki in sabka sum 1 ho. Yeh ensure karne ke liye, hum aakhri layer par final activation function ke roop mein `softmax` use karte hain. **Softmax** ek vector input leta hai aur yeh ensure karta hai ki us vector ke saare components probabilities mein convert ho jaayein.

Saath hi, kyunki network ka output $C$-dimensional vector hota hai, hume labels bhi usi form mein chahiye. Yeh **one-hot encoding** use karke kiya ja sakta hai, jisme class number $i$ ko zeros ke vector mein convert kiya jaata hai, aur $i$-th position par 1 rakha jaata hai.

Neural network ke probability output ko expected one-hot-encoded label se compare karne ke liye, hum **cross-entropy loss** function use karte hain. Yeh do probability distributions leta hai aur batata hai ki woh kitne alag hain.

Toh, $C$ classes ke saath multi-class classification ke liye hume yeh cheezein karni hongi:
* Network ki aakhri layer mein $C$ neurons hone chahiye
* Final activation function **softmax** hona chahiye
* Loss **cross-entropy loss** hona chahiye
* Labels ko **one-hot encoding** mein convert karna chahiye (yeh `numpy` use karke, ya Keras utils `to_categorical` use karke kiya ja sakta hai)

In [ ]:
model = keras.models.Sequential([
    keras.layers.Dense(5,input_shape=(2,),activation='relu'),
    keras.layers.Dense(2,activation='softmax')
])
model.compile(keras.optimizers.Adam(0.01),'categorical_crossentropy',['acc'])

# Two ways to convert to one-hot encoding
train_labels_onehot = keras.utils.to_categorical(train_labels)
test_labels_onehot = np.eye(2)[test_labels]

hist = model.fit(x=train_x_norm,y=train_labels_onehot,
                 validation_data=[test_x_norm,test_labels_onehot],batch_size=1,epochs=10)

### Sparse Categorical Cross-Entropy

Aksar multi-class classification mein labels ko class number se represent kiya jaata hai. Keras ek aur type ke loss function ko support karta hai jise **sparse categorical cross-entropy** kehte hain, jo class numbers ko integers ke roop mein expect karta hai, one-hot vectors ke roop mein nahi. Is type ke loss function ko use karke, hum apna training code simple bana sakte hain:

In [ ]:
model.compile(keras.optimizers.Adam(0.01),'sparse_categorical_crossentropy',['acc'])
model.fit(x=train_x_norm,y=train_labels,validation_data=[test_x_norm,test_labels],batch_size=1,epochs=10)

## Multi-label Classification

Kabhi-kabhi aise cases hote hain jab hamare objects ek saath do classes se belong kar sakte hain. Jaise maan lijiye hum ek aisa classification model develop karna chahte hain jo image mein cats aur dogs ko identify kar sake, lekin hum aise cases bhi allow karna chahte hain jahan image mein cat aur dog dono maujood hon.

Multi-label classification mein, one-hot encoded vector ke bajaye, hamare paas ek aisa vector hoga jismein un sabhi classes ke liye 1 hoga jo input sample se belong karti hain. Is tarah, network ka output saari classes ke liye normalized probabilities ke bajaye har class ke liye individually probabilities dega - jo **sigmoid** activation function use karne ke barabar hai. Cross-entropy loss ko ab bhi loss function ke roop mein use kiya ja sakta hai.

> **Note** ki yeh har particular class ke liye **alag-alag neural networks** use karke binary classification karne jaisa hi hai - bas network ka shuruaati hissa (final classification layer tak) saari classes ke liye shared hota hai.

## Classification Loss Functions ka Summary

Humne dekha ki binary, multi-class aur multi-label classification loss function aur network ki aakhri layer par activation function ke type se alag hote hain. Agar aap abhi seekhna shuru kar rahe hain, toh yeh thoda confusing lag sakta hai, lekin yaad rakhne ke liye yahan kuch rules hain:
* Agar network ka ek output hai (**binary classification**), toh hum **sigmoid** activation function use karte hain, aur **multi-class classification** ke liye - **softmax**.
* Agar output class ko one-hot-encoding ke roop mein represent kiya gaya hai, toh loss function **cross entropy loss** (categorical cross-entropy) hoga. Agar output mein class number hai - **sparse categorical cross-entropy**. **Binary classification** ke liye - **binary cross-entropy** (jo **log loss** ke jaisa hi hai) use karein.
* **Multi-label classification** woh hai jab koi object ek hi time par kai classes se belong kar sakta hai. Is case mein, hume labels ko one-hot encoding use karke encode karna hoga aur activation function ke roop mein **sigmoid** use karna hoga, taaki har class ki probability 0 aur 1 ke beech ho.

| Classification | Label Format | Activation Function | Loss |
|---------------|-----------------------|-----------------|----------|
| Binary      | Pehli class ki probability | Sigmoid | Binary cross-entropy |
| Binary      | One-hot encoding (2 outputs) | Softmax | Categorical cross-entropy |
| Multi-class |  One-hot encoding | Softmax | Categorical cross-entropy |
| Multi-class | Class number | Softmax | Sparse categorical cross-entropy |
| Multi-label | One-hot encoding | Sigmoid | Categorical cross-entropy |

**Task**:
Keras use karke MNIST handwritten digits ke liye ek classifier train karein:
* Dhyan dein ki Keras mein kuch standard datasets shaamil hain, jinmein MNIST bhi hai. Keras se MNIST use karne ke liye, aapko sirf kuch lines ka code chahiye (zyada jaankari [yahan](https://www.tensorflow.org/api_docs/python/tf/keras/datasets/mnist))
* Alag-alag network configurations try karein, jaise layers/neurons ki sankhya, activation functions.

Aapko sabse acchi accuracy kya mili?

## Main Takeaways

* **Keras** beginners ke liye bahut recommended hai, kyunki yeh layers se network ko aasaani se banane deta hai, aur phir ise sirf kuch lines ke code se train kiya ja sakta hai.
* Agar non-standard architectures ki zaroorat ho, toh aapko Tensorflow ko thoda gehraai se seekhna padega. Ya aap kisi se request kar sakte hain ki woh custom logic ko Keras layer ke roop mein implement kar de, aur phir ise Keras model mein use karein.
* PyTorch ko bhi dekhna aur approaches ko compare karna ek accha idea hai.

Keras aur Tensorflow 2.0 par Keras ke creator ka ek accha sample notebook [yahan](https://t.co/k694J95PI8) mil sakta hai.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->